# 第09回 進化と群知能：値とコードを一段ずつ読む

個体を選ぶ、経路の重みを更新する、候補位置へ移動する、という三つの探索を同じ初期状態から追います。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** 固定seedのGA・ACO・人工魚群が既存基準JSONと一致。更新前と更新後を一行ずつ説明できる。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
import argparse


In [ ]:
import copy


In [ ]:
import json


In [ ]:
import math


In [ ]:
SEED = 65535


In [ ]:
POPULATION = ['100000', '110000', '111000', '101010', '010101', '001111']


In [ ]:
LENGTHS = [4.0, 6.0, 8.0]


In [ ]:
POSITIONS = [[-1.5, -1.0], [-1.0, 1.0], [0.5, 1.5],
             [1.5, 0.5], [1.0, -1.5], [-0.5, -0.5]]


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 評価→選択→交叉→突然変異

**今いる段階：** 初期個体→親→子→次世代

**予想：** 交叉と突然変異は同じ操作か

**操作：** example_mutationを0.1から0へ変更し、同じセルを実行します。

**結果の目印：** 6個体の評価と親確率、交叉前後、変異前後が対応する

**戻す：** example_mutation=0.1に戻して再実行します。

**説明する：** 確率が高いことと必ず選ばれることの違い


### 計算の定義：`fitness`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def fitness(bits):
    if not isinstance(bits, str) or len(bits) != 6 or any(c not in '01' for c in bits):
        raise ValueError('個体は0と1からなる6文字です')
    return bits.count('1')


### 計算の定義：`finite`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def finite(value, low, high, name):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not low <= value <= high:
        raise ValueError(f'{name}は{low}〜{high}の有限数です')
    return value


### 計算の定義：`probabilities`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def probabilities(weights):
    if not weights:
        raise ValueError('重みを1個以上指定してください')
    for w in weights:
        finite(w, 0, 1e12, '重み')
    total = sum(weights)
    return [w / total for w in weights] if total else [1 / len(weights)] * len(weights)


### 計算の定義：`ga_evaluate`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def ga_evaluate(population):
    if len(population) != 6:
        raise ValueError('集団は6個体です')
    values = [fitness(x) for x in population]
    return {'fitness': values, 'probabilities': probabilities(values),
            'current_best': max(values), 'mean': sum(values) / 6,
            'diversity': len(set(population))}


### 計算の定義：`integer`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def integer(value, low, high, name):
    finite(value, low, high, name)
    if not isinstance(value, int):
        raise ValueError(f'{name}は整数です')
    return value


### 計算の定義：`crossover`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def crossover(a, b, cut):
    fitness(a)
    fitness(b)
    integer(cut, 1, 5, '交叉位置')
    return [a[:cut] + b[cut:], b[:cut] + a[cut:]]


### 計算の定義：`choose`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def choose(probs, rng):
    u, total = rng.random(), 0.0
    for i, p in enumerate(probs):
        total += p
        if u < total:
            return i
    return len(probs) - 1


### 計算の定義：`mutate`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def mutate(bits, rate, rng):
    fitness(bits)
    finite(rate, 0, 1, '突然変異率')
    changed = []
    for bit in bits:
        draw = rng.random()
        changed.append(str(1 - int(bit)) if draw < rate else bit)
    return ''.join(changed)


### 計算の定義：`ga_step`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def ga_step(population, rng, mutation=0.1):
    finite(mutation, 0, 1, '突然変異率')
    ev = ga_evaluate(population)
    elite = ev['fitness'].index(ev['current_best'])
    children, trace = [population[elite]], []
    while len(children) < 6:
        parents = [choose(ev['probabilities'], rng), choose(ev['probabilities'], rng)]
        cut = 1 + int(5 * rng.random())
        crossed = crossover(population[parents[0]], population[parents[1]], cut)
        # 最後の組でも2個体分の乱数を消費し、超過する子だけを捨てる。
        mutated = [mutate(x, mutation, rng) for x in crossed]
        trace.append({'parents': parents, 'cut': cut, 'crossed': crossed, 'mutated': mutated})
        children.extend(mutated[:6 - len(children)])
    return {'population': children, 'elite_index': elite, 'trace': trace, **ga_evaluate(children)}


### 計算の定義：`RNG`

教材用32bit LCG。呼出しごとに状態を進め、[0,1)を返す。


In [ ]:
class RNG:
    """教材用32bit LCG。呼出しごとに状態を進め、[0,1)を返す。"""
    def __init__(self, seed=SEED):
        self.state = integer(seed, 0, 2**32 - 1, 'シード')

    def random(self):
        self.state = (1664525 * self.state + 1013904223) % 2**32
        return self.state / 2**32


### 計算の定義：`lesson_ga`

同じ6個体を評価し、親の選択→交叉→突然変異→次世代を追う。


In [ ]:
def lesson_ga(mutation=0.1):
    """同じ6個体を評価し、親の選択→交叉→突然変異→次世代を追う。"""
    population = list(POPULATION)
    evaluation = ga_evaluate(population)
    result = ga_step(population,RNG(),mutation)
    rows = [(i,bits,evaluation['fitness'][i],evaluation['probabilities'][i],i==result['elite_index']) for i,bits in enumerate(population)]
    crosses = []
    for pair,item in enumerate(result['trace']):
        parent_a,parent_b = item['parents']
        for child in range(2):
            crosses.append((pair,population[parent_a],population[parent_b],item['cut'],item['crossed'][child],item['mutated'][child]))
    next_rows = [(i,bits,fitness(bits)) for i,bits in enumerate(result['population'])]
    return {'title':'GA：6個体の次世代を作る','question':'評価が高い個体だけが親になるか',
            'input':population,'columns':['番号','個体','1の個数=評価','親になる確率','エリート保存'],'rows':rows,
            'tables':[('同じ親から交叉→突然変異',['組','親A','親B','切る位置','交叉後','突然変異後'],crosses),('採用する6個体',['次世代番号','個体','評価'],next_rows)],
            'result':f"次世代{result['population']}",
            'comparison':f"同じseed・突然変異0の次世代{ga_step(population,RNG(),0)['population']}",
            'explanation':'評価を合計で割って選択確率にする。親は復元抽出。エリート1個と子5個を採用し、最後の組の余分な子は捨てる。',
            'reset':'mutation=0.1で再実行。同じ初期集団とseedから開始する。'}


In [ ]:
example_mutation = 0.1
program_observation_ga = lesson_ga(example_mutation)
learning_show(program_observation_ga, reset='example_mutation=0.1に戻して再実行します。')


## 2. 選択確率とフェロモン更新

**今いる段階：** 選択→通過数→蒸発→加算

**予想：** 蒸発率を変えると今回の選択も変わるか

**操作：** example_evaporationを0.25から0.5へ変更し、同じセルを実行します。

**結果の目印：** 同じseedなら今回の通過数は同じ、更新後τが変わる

**戻す：** example_evaporation=0.25に戻して再実行します。

**説明する：** 蒸発と加算はどの式で行われたか


### 計算の定義：`aco_probabilities`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def aco_probabilities(tau, lengths=LENGTHS):
    if len(tau) != 3 or len(lengths) != 3:
        raise ValueError('経路は3本です')
    for d in lengths:
        finite(d, 0.1, 100, '経路長')
    for t in tau:
        finite(t, 0, 1e6, 'フェロモン')
    return probabilities([t / d for t, d in zip(tau, lengths)])


### 計算の定義：`aco_update`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def aco_update(tau, counts, evaporation=0.25, lengths=LENGTHS):
    aco_probabilities(tau, lengths)
    finite(evaporation, 0, 1, '蒸発率')
    if len(counts) != 3:
        raise ValueError('通過数は3本分です')
    for count in counts:
        integer(count, 0, 6, '通過数')
    if sum(counts) > 6:
        raise ValueError('蟻は最大6匹です')
    evaporated, deposits = [], []
    for pheromone, count, length in zip(tau, counts, lengths):
        evaporated.append((1 - evaporation) * pheromone)
        deposits.append(6 * count / length)
    return {'evaporated': evaporated, 'deposits': deposits,
            'tau': [t + d for t, d in zip(evaporated, deposits)]}


### 計算の定義：`aco_step`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def aco_step(tau, rng, evaporation=0.25, lengths=LENGTHS):
    finite(evaporation, 0, 1, '蒸発率')
    probs = aco_probabilities(tau, lengths)
    choices = [choose(probs, rng) for _ in range(6)]
    counts = [choices.count(i) for i in range(3)]
    return {'probabilities': probs, 'choices': choices, 'counts': counts,
            'current_best': min(lengths[i] for i in choices),
            **aco_update(tau, counts, evaporation, lengths)}


### 計算の定義：`lesson_aco`

経路を選ぶ確率と、選んだ後の蒸発・加算を分ける。


In [ ]:
def lesson_aco(evaporation=0.25):
    """経路を選ぶ確率と、選んだ後の蒸発・加算を分ける。"""
    tau = [1.,1.,1.]
    result = aco_step(tau,RNG(),evaporation)
    weights = [t/d for t,d in zip(tau,LENGTHS)]
    rows = [(i,LENGTHS[i],tau[i],weights[i],result['probabilities'][i],result['counts'][i],result['evaporated'][i],result['deposits'][i],result['tau'][i]) for i in range(3)]
    return {'title':'ACO：選択→蒸発→通過数に応じた加算','question':'短い経路はどの計算で有利になるか',
            'input':f'経路長{LENGTHS}、初期フェロモン{tau}、6匹、蒸発率{evaporation}',
            'columns':['経路','長さ','初期τ','τ÷長さ','選択確率','通過数','蒸発後','6×通過数÷長さ','更新後'],'rows':rows,
            'tables':[('各蟻の選択（同じ初期確率から6回選ぶ）',['蟻','選んだ経路'],list(enumerate(result['choices'])))],
            'result':f"更新後τ={result['tau']}; 次回の確率={aco_probabilities(result['tau'])}",
            'comparison':f"蒸発率0.5でのτ={aco_step(tau,RNG(),0.5)['tau']}",
            'explanation':'選択した6匹の通過数を集計してから、一度だけ蒸発と加算を行う。今回選択する確率と次回の確率を混ぜない。',
            'reset':'evaporation=0.25で再実行。τは[1,1,1]へ戻る。'}


In [ ]:
example_evaporation = 0.25
program_observation_aco = lesson_aco(example_evaporation)
learning_show(program_observation_aco, reset='example_evaporation=0.25に戻して再実行します。')


## 3. 候補位置を評価して移動する

**今いる段階：** 近傍→候補→採否→移動

**予想：** 近傍が増えれば必ず改善するか

**操作：** example_visualを1.0から2.0へ変更し、同じセルを実行します。

**結果の目印：** 魚ごとの旧位置・候補評価・採否・新位置が並ぶ

**戻す：** example_visual=1.0に戻して再実行します。

**説明する：** 改善条件と混雑条件を両方読む理由


### 計算の定義：`random_target`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def random_target(point, visual, rng):
    # 半径→角度の順。半径一様（円内面積一様ではない）という教材仕様。
    radius, angle = visual * rng.random(), 2 * math.pi * rng.random()
    return [max(-2.0, min(2.0, point[0] + radius * math.cos(angle))),
            max(-2.0, min(2.0, point[1] + radius * math.sin(angle)))]


### 計算の定義：`objective`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def objective(point):
    if len(point) != 2:
        raise ValueError('位置はx,yの2値です')
    for v in point:
        finite(v, -2, 2, '座標')
    return point[0] ** 2 + point[1] ** 2


### 計算の定義：`distance`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def distance(a, b):
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))


### 計算の定義：`move_toward`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def move_toward(point, target, step=0.2):
    objective(point)
    objective(target)
    finite(step, 0, 1, '移動幅')
    d = distance(point, target)
    if d == 0:
        return list(point)
    scale = min(step, d) / d
    return [max(-2.0, min(2.0, a + scale * (b - a))) for a, b in zip(point, target)]


### 計算の定義：`fish_action`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def fish_action(points, index, rng, visual=1.0, step=0.2, crowd=2):
    if not 1 <= len(points) <= 30:
        raise ValueError('魚は1〜30匹です')
    for point in points:
        objective(point)
    integer(index, 0, len(points) - 1, '魚番号')
    finite(visual, 0, 4, '視野')
    finite(step, 0, 1, '移動幅')
    integer(crowd, 0, 29, '混雑上限')
    p = points[index]
    neighbors = [j for j, q in enumerate(points) if j != index and distance(p, q) <= visual]
    candidates = []
    if neighbors:
        best = min(neighbors, key=lambda j: objective(points[j]))
        center = [sum(points[j][k] for j in neighbors) / len(neighbors) for k in range(2)]
        for action, target in [('follow', points[best]), ('swarm', center)]:
            density = sum(j != index and distance(target, q) <= visual for j, q in enumerate(points))
            moved = move_toward(p, target, step)
            accepted = density <= crowd and objective(target) < objective(p) and objective(moved) < objective(p)
            candidates.append({'action': action, 'target': list(target), 'density': density,
                               'accepted': accepted, 'position': moved})
    accepted = [c for c in candidates if c['accepted']]
    if accepted:
        chosen = min(accepted, key=lambda c: objective(c['position']))
        return {**chosen, 'neighbors': neighbors, 'candidates': candidates, 'trials': []}
    trials = []
    for _ in range(3):
        target = random_target(p, visual, rng)
        trials.append(target)
        if objective(target) < objective(p):
            return {'action': 'prey', 'position': move_toward(p, target, step),
                    'neighbors': neighbors, 'candidates': candidates, 'trials': trials}
    target = random_target(p, visual, rng)
    return {'action': 'random', 'position': move_toward(p, target, step),
            'neighbors': neighbors, 'candidates': candidates, 'trials': trials}


### 計算の定義：`fish_step`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def fish_step(points, rng, visual=1.0, step=0.2, crowd=2):
    # 全魚の候補を同じ旧状態から作る。同一反復内の新位置を参照しない。
    actions = []
    for index in range(len(points)):
        action = fish_action(points, index, rng, visual, step, crowd)
        actions.append(action)
    return {'positions': [a['position'] for a in actions], 'actions': actions}


### 計算の定義：`lesson_fish`

同じ旧位置から各魚の候補・評価・移動を比較する。


In [ ]:
def lesson_fish(visual=1.0):
    """同じ旧位置から各魚の候補・評価・移動を比較する。"""
    points = copy.deepcopy(POSITIONS)
    result = fish_step(points,RNG(),visual=visual)
    rows, candidates = [], []
    for index,action in enumerate(result['actions']):
        old,new = points[index],action['position']
        rows.append((index,old,objective(old),action['neighbors'],action['action'],new,objective(new)))
        for candidate in action['candidates']:
            candidates.append((index,candidate['action'],candidate['target'],objective(candidate['target']),candidate['density'],candidate['accepted']))
        for target in action['trials']:
            candidates.append((index,'prey試行',target,objective(target),'—',objective(target)<objective(old)))
    return {'title':'人工魚群：候補の評価から移動を決める','question':'近くの魚へ必ず移動するか',
            'input':f'6匹、視野{visual}、移動幅0.2。目的値x²+y²は小さいほどよい。',
            'columns':['魚','旧位置','旧評価','近傍番号','選択行動','新位置','新評価'],'rows':rows,
            'tables':[('候補と判定（採用は改善と混雑条件で決まる）',['魚','候補の種類','候補位置','評価','混雑','受理'],candidates)],
            'result':f"今回の最良評価={min(objective(point) for point in result['positions'])}",
            'comparison':f"視野2の行動={[a['action'] for a in fish_step(points,RNG(),visual=2)['actions']]}",
            'explanation':'追尾・群れの候補を改善と混雑で判定し、使えなければ探索する。同じ反復では全魚が更新前の位置を参照する。単一seedで最適到達は保証しない。',
            'reset':'visual=1.0で再実行。6匹の初期位置とseedを戻す。'}


In [ ]:
example_visual = 1.0
program_observation_fish = lesson_fish(example_visual)
learning_show(program_observation_fish, reset='example_visual=1.0に戻して再実行します。')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`run_ga`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def run_ga(steps=20, mutation=0.1, seed=SEED, population=None):
    integer(steps, 0, 200, '世代数')
    finite(mutation, 0, 1, '突然変異率')
    pop = list(POPULATION if population is None else population)
    rng = RNG(seed)
    history = [{'step': 0, **ga_evaluate(pop)}]
    for step in range(1, steps + 1):
        result = ga_step(pop, rng, mutation)
        pop = result['population']
        history.append({'step': step, **ga_evaluate(pop)})
    return {'population': pop, 'history': history, 'best_seen': max(h['current_best'] for h in history)}


### 計算の定義：`run_aco`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def run_aco(steps=20, evaporation=0.25, seed=SEED, tau=None, lengths=None):
    integer(steps, 0, 200, '反復数')
    finite(evaporation, 0, 1, '蒸発率')
    current = list([1.0] * 3 if tau is None else tau)
    lengths = list(LENGTHS if lengths is None else lengths)
    aco_probabilities(current, lengths)
    rng, history, best = RNG(seed), [], None
    for step in range(1, steps + 1):
        r = aco_step(current, rng, evaporation, lengths)
        best = r['current_best'] if best is None else min(best, r['current_best'])
        current = r['tau']
        history.append({'step': step, **r, 'best_seen': best})
    return {'tau': current, 'history': history, 'best_seen': best}


### 計算の定義：`run_fish`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def run_fish(steps=20, visual=1.0, step=0.2, crowd=2, seed=SEED, positions=None):
    integer(steps, 0, 200, '反復数')
    finite(visual, 0, 4, '視野')
    finite(step, 0, 1, '移動幅')
    integer(crowd, 0, 29, '混雑上限')
    points = copy.deepcopy(POSITIONS if positions is None else positions)
    if not 1 <= len(points) <= 30:
        raise ValueError('魚は1〜30匹です')
    values = [objective(p) for p in points]
    best, best_position = min(values), list(points[values.index(min(values))])
    rng = RNG(seed)
    history = [{'step': 0, 'current_best': best, 'best_seen': best}]
    for iteration in range(1, steps + 1):
        result = fish_step(points, rng, visual, step, crowd)
        points = result['positions']
        values = [objective(p) for p in points]
        current = min(values)
        if current < best:
            best, best_position = current, list(points[values.index(current)])
        history.append({'step': iteration, 'current_best': current, 'best_seen': best})
    return {'positions': points, 'history': history, 'best_seen': best, 'best_position': best_position}


### 計算の定義：`rounded`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def rounded(value):
    if isinstance(value, float):
        return round(value, 6)
    if isinstance(value, list):
        return [rounded(x) for x in value]
    if isinstance(value, dict):
        return {k: rounded(v) for k, v in value.items()}
    return value


### 計算の定義：`demo`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def demo():
    return rounded({'ga_initial': ga_evaluate(POPULATION),
                    'ga_step': ga_step(POPULATION, RNG()), 'ga': run_ga(),
                    'aco_step': aco_step([1.0] * 3, RNG()), 'aco': run_aco(),
                    'fish_step': fish_step(POSITIONS, RNG()), 'fish': run_fish()})


## 準備・固定入力・乱数


## GA：集団を更新する


## ACO：経路を選択して一括更新する


## AFSA：教材用の行動と反復


## 基準値を記録する


In [ ]:
baseline_result = demo()
print('基準：GA', baseline_result['ga']['best_seen'], 'ACO', baseline_result['aco']['best_seen'], 'AFSA', baseline_result['fish']['best_seen'])
print('GA初期確率:', baseline_result['ga_initial']['probabilities'])
print('ACO最初の更新:', baseline_result['aco_step']['tau'])


## 1 ビット列を変える

**問い・予想：先頭を111111にすると、その選択確率はどうなる？** 実行前に下のメモへ予想を書きます。

予想：

手順：次のセルを実行し、変更前後の値を比較してください。


In [ ]:
trial_population = list(POPULATION)
trial_population[0] = '111111'
trial_selection = ga_evaluate(trial_population)
print('先頭の選択確率:', baseline_result['ga_initial']['probabilities'][0], '→', trial_selection['probabilities'][0])


**想定結果・説明する問い：** 1/16から6/21へ増えます。他の個体の確率も分母の変化により変わります。

観察値：

規則に基づく説明：


## 2 突然変異率だけを変える

**問い・予想：0にすると最良と平均はどうなる？** 実行前に下のメモへ予想を書きます。

予想：

手順：次のセルを実行し、変更前後の値を比較してください。


In [ ]:
trial_ga = run_ga(mutation=0)
print('基準の最終集団:', baseline_result['ga']['population'])
print('変更後の最終集団:', trial_ga['population'])
print('基準と変更後の平均:', baseline_result['ga']['history'][-1]['mean'], trial_ga['history'][-1]['mean'])


**想定結果・説明する問い：** 最良値だけでなく集団と平均を比較します。反転をなくしても交叉は残ります。

観察値：

規則に基づく説明：


## 3 初期フェロモンだけを変える

**問い・予想：Bを1から2にすると選択確率は？** 実行前に下のメモへ予想を書きます。

予想：

手順：次のセルを実行し、変更前後の値を比較してください。


In [ ]:
trial_tau = [1.0, 2.0, 1.0]
trial_aco_prob = aco_probabilities(trial_tau)
print('基準:', baseline_result['aco_step']['probabilities'])
print('変更:', trial_aco_prob)


**想定結果・説明する問い：** Bは4/13から8/17へ変わります。長さは変更していません。

観察値：

規則に基づく説明：


## 4 蒸発率だけを変える

**問い・予想：最初の選択まで変わる？** 実行前に下のメモへ予想を書きます。

予想：

手順：次のセルを実行し、変更前後の値を比較してください。


In [ ]:
trial_aco = aco_step([1.0] * 3, RNG(), evaporation=0.5)
print('通過数:', trial_aco['counts'])
print('更新後:', trial_aco['tau'])


**想定結果・説明する問い：** 通過数は[2,3,1]のまま、更新後は[3.5,3.5,1.25]です。選択にはまだ蒸発率を使いません。

観察値：

規則に基づく説明：


## 5 視野だけを変える

**問い・予想：視野2なら、先頭の魚は何を選ぶ？** 実行前に下のメモへ予想を書きます。

予想：

手順：次のセルを実行し、変更前後の値を比較してください。


In [ ]:
trial_visual = fish_step(copy.deepcopy(POSITIONS), RNG(), visual=2)
print('基準の行動:', [a['action'] for a in baseline_result['fish_step']['actions']])
print('変更の行動:', [a['action'] for a in trial_visual['actions']])
print('先頭の候補:', trial_visual['actions'][0]['candidates'])


**想定結果・説明する問い：** 近傍と混雑数の両方が変わります。acceptedとdensityを読んで、候補の採否を説明してください。

観察値：

規則に基づく説明：


## 6 移動幅だけを変える

**問い・予想：0.4なら必ずよくなる？** 実行前に下のメモへ予想を書きます。

予想：

手順：次のセルを実行し、変更前後の値を比較してください。


In [ ]:
trial_step = run_fish(step=0.4)
print('基準の探索中最良:', baseline_result['fish']['best_seen'])
print('変更の探索中最良:', trial_step['best_seen'])
print('基準は保持:', baseline_result == demo())


**想定結果・説明する問い：** 同じシードの2条件を比べた結果です。今回よくなっても、一般的に大きいほどよいとは言えません。

観察値：

規則に基づく説明：


## リセット・困ったとき

元ファイルを残し、変更用のコピーで作業してください。元に戻すには基準ファイルを開き、カーネル再起動→全セル実行します。

- `NameError`：定義セルを飛ばしていないか確認し、全セル実行します。
- カーネルがない：VS Codeの右上で既存venvを選びます。必要ならそのvenvに同梱requirements.txtを使ってJupyter環境を用意します。
- `ValueError`：入力範囲を確認します。NaN・無限大、反復201以上、6ビット以外は使えません。
- 結果が違う：入力・シード・反復数・セル実行順を確認してください。基準のGA=6、ACO=4、AFSA=0.000632は小数点以下6桁の値です。

レポートの形式・締切・提出先は別の授業案内に従ってください。このNotebookから外部への送信は行いません。


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python evolutionary_computation.py --explain all`で同じ計算表を表示します。各テーマは`--explain ga`のように指定できます。従来のコマンドと固定期待値も維持しています。
